In [2]:
import pandas as pd
import sys
sys.path.insert(0, "/Users/ruturajdesai/Desktop/Data-Science/projects/end_to_end/aml_detection")
from aml_detection.splitting import load_split

rows = {}
for name in ["train", "val", "test"]:
    d = load_split(name)
    fmt = d.groupby("payment_format", observed=True).is_laundering.agg(["sum", "size"])
    rows[name] = {
        "rows": len(d),
        "positives": int(d.is_laundering.sum()),
        "rate": d.is_laundering.mean(),
        "ach_share_of_positives": fmt.loc["ACH", "sum"] / d.is_laundering.sum(),
        "ach_share_of_rows": fmt.loc["ACH", "size"] / len(d),
    }

print(pd.DataFrame(rows).T.round(5).to_string())

2026-08-22 15:34:05.756 | INFO     | aml_detection.config:<module>:11 - PROJ_ROOT path is: /Users/ruturajdesai/Desktop/Data-Science/projects/end_to_end/aml_detection


2026-08-22 15:34:06.488 | WARNING  | aml_detection.splitting:load_split:250 - Loading the TEST split. This should happen only in final evaluation -- use 'val' for tuning and threshold selection.
            rows  positives     rate  ach_share_of_positives  ach_share_of_rows
train  1109155.0     1563.0  0.00141                 0.88292            0.11924
val     954954.0     1081.0  0.00113                 0.85384            0.11180
test   1015418.0     1143.0  0.00113                 0.87402            0.13611


In [4]:
import mlflow
mlflow.set_tracking_uri(
    "file:///Users/ruturajdesai/Desktop/Data-Science/projects/end_to_end/aml_detection/mlruns"
)

In [5]:
runs = mlflow.search_runs(experiment_names=["aml_detection"])
print(runs.shape)
print([c for c in runs.columns if "run" in c.lower() or "name" in c.lower()])

Traceback (most recent call last):
  File "/Users/ruturajdesai/Desktop/Data-Science/projects/end_to_end/aml_detection/.venv/lib/python3.11/site-packages/mlflow/store/tracking/file_store.py", line 386, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/ruturajdesai/Desktop/Data-Science/projects/end_to_end/aml_detection/.venv/lib/python3.11/site-packages/mlflow/store/tracking/file_store.py", line 487, in _get_experiment
    meta = FileStore._read_yaml(experiment_dir, FileStore.META_DATA_FILE_NAME)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/ruturajdesai/Desktop/Data-Science/projects/end_to_end/aml_detection/.venv/lib/python3.11/site-packages/mlflow/store/tracking/file_store.py", line 1676, in _read_yaml
    return _read_helper(root, file_name, attempts_remaining=retries)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/

(0, 6)
['run_id']


In [3]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
import mlflow
from sklearn.metrics import average_precision_score
from aml_detection.features import feature_matrix, TARGET

test = load_split("test")
X, y = feature_matrix(test), test[TARGET]

runs = mlflow.search_runs(experiment_names=["aml_detection"])
for name in ["xgboost_weighted", "cv_xgboost"]:
    r = runs[runs["tags.mlflow.runName"] == name].iloc[0]
    m = mlflow.sklearn.load_model(f"runs:/{r.run_id}/model")
    test_pr = average_precision_score(y, m.predict_proba(X)[:, 1])
    print(f"{name:20} val {r['metrics.pr_auc']:.4f} -> test {test_pr:.4f} "
          f"(drop {r['metrics.pr_auc'] - test_pr:+.4f})")

2026-08-22 15:36:03.110 | WARNING  | aml_detection.splitting:load_split:250 - Loading the TEST split. This should happen only in final evaluation -- use 'val' for tuning and threshold selection.


2026/08/22 15:36:04 WARNING mlflow.tracking.fluent: Cannot retrieve experiment by name aml_detection


KeyError: 'tags.mlflow.runName'